# Step 7 — Robustness across model families

Compares model families on the deployable `H2` setting with weather at both endpoints (paper Sect. 5.4, Table 9), to show that the conclusions do not depend on the choice of backbone.

- **Input:** `data/processed/flights_clean.parquet`, `data/processed/wx_H2.parquet`
- **Output:** `results/results_model_comparison.csv`

| Model | Notes |
|---|---|
| Constant: always 0, median, mean | Reference baselines |
| Linear / logistic regression | Median imputation and standardisation |
| Random forest | Trained on a 1M-row subsample |
| LightGBM (L2, Huber, L1) | L1/Huber target MAE |
| XGBoost | Early stopping on validation |

### Configuration
`SETTINGS_CMP` selects the settings to compare (default `['H2']`); set `XGB_DEVICE = 'cpu'` if no GPU is available. Use 0.1 for a quick test run

In [1]:
import time, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, roc_auc_score,
                             average_precision_score, precision_recall_curve, f1_score)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'README.md').exists())  # project root
warnings.filterwarnings('ignore')

DATA = ROOT / 'data' / 'processed'
RESULTS = ROOT / 'results'; RESULTS.mkdir(parents=True, exist_ok=True)
SETTINGS_CMP = ['H2']            # add 'oracle' to compare the gap for every model
SEED = 42
SAMPLE = 0.1                     # use 0.1 for a quick test run
RF_N = 1_000_000                 # random forest is trained on a subsample
XGB_DEVICE = 'cuda'              # set to 'cpu' if no GPU is available
N_BOOT = 1000
TRAIN_END, VAL_END = '2022-01-01', '2022-05-01'

### Load data, chronological split, and out-of-fold target encoding
Identical to `05_modeling_horizon.ipynb`; categorical columns are integer-coded so that all models share one feature matrix.

In [2]:
BASE_NUM = ['Month', 'DayOfWeek', 'CRSDepHour', 'CRSArrHour', 'CRSElapsedTime', 'Distance',
            'DistanceGroup', 'DepHour_sin', 'DepHour_cos', 'Month_sin', 'Month_cos',
            'DoW_sin', 'DoW_cos', 'IsWeekend', 'IsPeakHour', 'IsHoliday']
HIGH_CARD = ['Origin', 'Dest', 'Airline', 'Tail_Number']
LOW_CARD  = ['OriginState', 'DestState', 'DepTimeBlk']

df = pd.read_parquet(DATA / 'flights_clean.parquet',
                     columns=['row_id', 'FlightDate', 'ArrDelayMinutes', 'ArrDel15'] + BASE_NUM + HIGH_CARD + LOW_CARD)
assert (df['row_id'].values == np.arange(len(df))).all()
tr = (df['FlightDate'] < TRAIN_END).values
va = ((df['FlightDate'] >= TRAIN_END) & (df['FlightDate'] < VAL_END)).values
te = (df['FlightDate'] >= VAL_END).values
if SAMPLE < 1.0:
    tr = tr & (np.random.RandomState(0).rand(len(df)) < SAMPLE)
print(f'Train {tr.sum():,} | Val {va.sum():,} | Test {te.sum():,}')

y_reg = df['ArrDelayMinutes'].astype('float32').values
y_reg = np.minimum(y_reg, np.quantile(y_reg[tr], 0.999))
y_cls = df['ArrDel15'].astype('int8').values


def target_encode_oof(col, y, m=10.0, n_folds=5, seed=0):
    prior = y[tr].mean()
    out = np.full(len(col), np.nan, dtype='float32')
    vals = col.values
    def fit_apply(fit_idx, apply_idx):
        s = pd.DataFrame({'k': vals[fit_idx], 'y': y[fit_idx]}).groupby('k')['y'].agg(['sum', 'count'])
        enc = (s['sum'] + m * prior) / (s['count'] + m)
        out[apply_idx] = pd.Series(vals[apply_idx]).map(enc).fillna(prior).values
    tr_idx = np.where(tr)[0]
    for f_idx, a_idx in KFold(n_folds, shuffle=True, random_state=seed).split(tr_idx):
        fit_apply(tr_idx[f_idx], tr_idx[a_idx])
    fit_apply(tr_idx, np.where(~tr)[0])
    return out


X_base = df[BASE_NUM].astype('float32').copy()
for c in HIGH_CARD:
    X_base[f'{c}_te'] = target_encode_oof(df[c].astype(str), y_cls.astype('float32'))
for c in LOW_CARD:
    X_base[c] = df[c].astype('category').cat.codes.astype('float32')
dates_test = df.loc[te, 'FlightDate'].values
del df; gc.collect()
print('X_base:', X_base.shape)

Train 855,049 | Val 792,335 | Test 638,672
X_base: (9981331, 23)


### Evaluation helpers
Regression: MAE, RMSE, R²; classification: ROC-AUC, PR-AUC, and F1 at the validation-tuned threshold; day-level bootstrap for paired comparisons.

In [3]:
def eval_reg(p):
    return dict(MAE=mean_absolute_error(y_reg[te], p), RMSE=np.sqrt(mean_squared_error(y_reg[te], p)),
                R2=r2_score(y_reg[te], p))


def eval_cls(p_val, p_test):
    prec, rec, thr = precision_recall_curve(y_cls[va], p_val)
    f1v = 2 * prec * rec / (prec + rec + 1e-9)
    t = thr[np.argmax(f1v[:-1])]
    return dict(AUC=roc_auc_score(y_cls[te], p_test), PR_AUC=average_precision_score(y_cls[te], p_test),
                F1=f1_score(y_cls[te], p_test > t), thr=t)


def per_day(p_reg, p_cls):
    d = pd.DataFrame({'d': dates_test,
                      'ae': np.abs(y_reg[te] - p_reg) if p_reg is not None else np.nan,
                      'br': (p_cls - y_cls[te]) ** 2 if p_cls is not None else np.nan})
    return d.groupby('d').agg(['sum', 'count'])


def boot_ci(day_a, day_b, key, n_boot=N_BOOT, seed=0):
    # 95% CI of (metric_a - metric_b); key = 'ae' (MAE) or 'br' (Brier)
    idx = np.random.RandomState(seed).randint(0, len(day_a), size=(n_boot, len(day_a)))
    sa, sb, c = day_a[(key, 'sum')].values, day_b[(key, 'sum')].values, day_a[(key, 'count')].values
    d = (sa[idx].sum(1) - sb[idx].sum(1)) / c[idx].sum(1)
    return np.percentile(d, 2.5), np.percentile(d, 97.5)

### Model definitions
Random forest: 200 trees, `max_depth=20`, `min_samples_leaf=50`; XGBoost: `max_depth=8`, `learning_rate=0.03`; LightGBM as in `05_modeling_horizon.ipynb`.

In [4]:
def run_all_models(X):
    preds = {}
    Xtr, Xva, Xte = X[tr], X[va], X[te]

    # --- Linear / Logistic (impute + scale)
    t = time.time()
    med = Xtr.median()
    sc = StandardScaler().fit(Xtr.fillna(med))
    Ztr, Zva, Zte = (sc.transform(Z.fillna(med)) for Z in (Xtr, Xva, Xte))
    lin = LinearRegression().fit(Ztr, y_reg[tr])
    lr = LogisticRegression(max_iter=300).fit(Ztr, y_cls[tr])
    preds['Linear / Logistic'] = (lin.predict(Zte), lr.predict_proba(Zva)[:, 1], lr.predict_proba(Zte)[:, 1])
    print(f'  Linear/Logistic xong ({time.time()-t:.0f}s)'); del Ztr, Zva, Zte

    # --- Random forest (subsample)
    t = time.time()
    rs = np.random.RandomState(SEED)
    sub = rs.choice(len(Xtr), size=min(RF_N, len(Xtr)), replace=False)
    Xs = Xtr.iloc[sub].fillna(med)
    rf_kw = dict(n_estimators=200, max_depth=20, min_samples_leaf=50, max_features='sqrt', n_jobs=-1, random_state=SEED)
    rfr = RandomForestRegressor(**rf_kw).fit(Xs, y_reg[tr][sub])
    rfc = RandomForestClassifier(class_weight='balanced', **rf_kw).fit(Xs, y_cls[tr][sub])
    preds['Random Forest (1M)'] = (rfr.predict(Xte.fillna(med)), rfc.predict_proba(Xva.fillna(med))[:, 1],
                                   rfc.predict_proba(Xte.fillna(med))[:, 1])
    print(f'  RF xong ({time.time()-t:.0f}s)'); del rfr, rfc, Xs

    # --- LightGBM (L2 and Huber for regression; binary for classification)
    base = dict(learning_rate=0.05, num_leaves=255, min_data_in_leaf=200, feature_fraction=0.8,
                bagging_fraction=0.8, bagging_freq=5, lambda_l2=1.0, verbose=-1, seed=SEED, num_threads=-1)
    def lgb_fit(obj, y, extra=None):
        p = dict(base, objective=obj, **(extra or {}))
        m = lgb.train(p, lgb.Dataset(Xtr, label=y[tr]), num_boost_round=3000,
                      valid_sets=[lgb.Dataset(Xva, label=y[va])],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        return m
    t = time.time()
    m_l2 = lgb_fit('regression', y_reg, dict(metric='l1'))
    m_hb = lgb_fit('huber', y_reg, dict(metric='l1', alpha=30.0))
    m_c = lgb_fit('binary', y_cls, dict(metric='binary_logloss'))
    pc_va, pc_te = m_c.predict(Xva, num_iteration=m_c.best_iteration), m_c.predict(Xte, num_iteration=m_c.best_iteration)
    preds['LightGBM (L2)'] = (m_l2.predict(Xte, num_iteration=m_l2.best_iteration), pc_va, pc_te)
    preds['LightGBM (Huber)'] = (m_hb.predict(Xte, num_iteration=m_hb.best_iteration), None, None)
    print(f'  LightGBM xong ({time.time()-t:.0f}s)')

    # --- XGBoost
    t = time.time()
    xp = dict(tree_method='hist', device=XGB_DEVICE, max_depth=8, learning_rate=0.03, subsample=0.8,
              colsample_bytree=0.8, reg_lambda=1.0, min_child_weight=100, seed=SEED, verbosity=0)
    dtr_r, dva_r = xgb.DMatrix(Xtr, label=y_reg[tr]), xgb.DMatrix(Xva, label=y_reg[va])
    xr = xgb.train(dict(xp, objective='reg:squarederror', eval_metric='mae'), dtr_r, 3000,
                   evals=[(dva_r, 'val')], early_stopping_rounds=100, verbose_eval=False)
    dtr_c, dva_c = xgb.DMatrix(Xtr, label=y_cls[tr]), xgb.DMatrix(Xva, label=y_cls[va])
    xc = xgb.train(dict(xp, objective='binary:logistic', eval_metric='logloss'), dtr_c, 3000,
                   evals=[(dva_c, 'val')], early_stopping_rounds=100, verbose_eval=False)
    dte = xgb.DMatrix(Xte)
    rng_r, rng_c = (0, xr.best_iteration + 1), (0, xc.best_iteration + 1)
    preds['XGBoost'] = (xr.predict(dte, iteration_range=rng_r), xc.predict(dva_c, iteration_range=rng_c),
                        xc.predict(dte, iteration_range=rng_c))
    print(f'  XGBoost xong ({time.time()-t:.0f}s)')
    return preds

### Run all models and compare with LightGBM
`dMAE_vs_LGB` and `dBrier_vs_LGB` give 95% day-level bootstrap intervals of the difference to LightGBM (L2).

In [5]:
rows = []
for name, const in [('Const: always 0', 0.0), ('Const: median', float(np.median(y_reg[tr]))),
                    ('Const: mean', float(y_reg[tr].mean()))]:
    rows.append(dict(setting='-', model=name, **eval_reg(np.full(te.sum(), const, dtype='float32'))))

for setting in SETTINGS_CMP:
    print(f'\n=== Setting {setting} ===')
    W = pd.read_parquet(DATA / f'wx_{setting}.parquet')
    assert (W['row_id'].values == np.arange(len(W))).all()
    X = pd.concat([X_base, W.drop(columns='row_id').astype('float32')], axis=1)
    del W
    preds = run_all_models(X)
    del X; gc.collect()

    ref = per_day(preds['LightGBM (L2)'][0], preds['LightGBM (L2)'][2])
    for name, (p_reg, p_va, p_te) in preds.items():
        r = dict(setting=setting, model=name, **eval_reg(p_reg))
        if p_te is not None:
            r.update(eval_cls(p_va, p_te))
        if name != 'LightGBM (L2)':
            day = per_day(p_reg, p_te)
            r['dMAE_vs_LGB_lo'], r['dMAE_vs_LGB_hi'] = boot_ci(day, ref, 'ae')
            if p_te is not None:
                r['dBrier_vs_LGB_lo'], r['dBrier_vs_LGB_hi'] = boot_ci(day, ref, 'br')
        rows.append(r)

cmp_df = pd.DataFrame(rows)
cmp_df.to_csv(RESULTS / 'results_model_comparison.csv', index=False)
cmp_df.round(4)


=== Setting H2 ===
  Linear/Logistic xong (11s)
  RF xong (88s)
  LightGBM xong (72s)
  XGBoost xong (18s)


,setting,model,MAE,RMSE,R2,AUC,PR_AUC,F1,thr,dMAE_vs_LGB_lo,dMAE_vs_LGB_hi,dBrier_vs_LGB_lo,dBrier_vs_LGB_hi
0,-,Const: always 0,16.4263,47.3486,-0.1368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-,Const: median,16.4263,47.3486,-0.1368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-,Const: mean,20.7681,44.6184,-0.0095,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,H2,Linear / Logistic,20.5414,43.8307,0.0258,0.6594,0.3663,0.4287,0.1454,0.0992,0.4088,0.0036,0.0056
4,H2,Random Forest (1M),20.1848,43.4960,0.0407,0.6824,0.3971,0.4392,0.4440,-0.2262,0.0059,0.0538,0.0644
5,H2,LightGBM (L2),20.2933,43.4449,0.0429,0.6823,0.3999,0.4376,0.1410,NaN,NaN,NaN,NaN
6,H2,LightGBM (Huber),17.6893,44.3641,0.0020,NaN,NaN,NaN,NaN,-2.7732,-2.4259,NaN,NaN
7,H2,XGBoost,20.2764,43.3855,0.0455,0.6813,0.3970,0.4358,0.1446,-0.0486,0.0158,0.0002,0.0010


### LightGBM with L1 loss vs. the constant-zero baseline
MAE-optimised LightGBM on `H2`, compared with always predicting 0 (paired day-level bootstrap).

In [6]:
# --- LightGBM with L1 loss (MAE-optimal) on H2 vs. the constant-zero baseline ---
W = pd.read_parquet(DATA / 'wx_H2.parquet').drop(columns='row_id').astype('float32')
X = pd.concat([X_base, W], axis=1); del W

p = dict(objective='regression_l1', metric='l1', learning_rate=0.05, num_leaves=255,
         min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
         lambda_l2=1.0, verbose=-1, seed=SEED, num_threads=-1)
m_l1 = lgb.train(p, lgb.Dataset(X[tr], label=y_reg[tr]), num_boost_round=3000,
                 valid_sets=[lgb.Dataset(X[va], label=y_reg[va])],
                 callbacks=[lgb.early_stopping(100, verbose=False)])
pred_l1 = m_l1.predict(X[te], num_iteration=m_l1.best_iteration)
del X; gc.collect()

r = dict(setting='H2', model='LightGBM (L1)', **eval_reg(pred_l1))
# 95% CI of MAE(L1) - MAE(zero); both bounds < 0 => L1 significantly better
lo, hi = boot_ci(per_day(pred_l1, None), per_day(np.zeros(te.sum()), None), 'ae')
r['dMAE_vs_zero_lo'], r['dMAE_vs_zero_hi'] = lo, hi
print(r)

cmp_df = pd.concat([cmp_df, pd.DataFrame([r])], ignore_index=True)
cmp_df.to_csv(RESULTS / 'results_model_comparison.csv', index=False)

{'setting': 'H2', 'model': 'LightGBM (L1)', 'MAE': 16.22891051402658, 'RMSE': np.float64(46.36399825530899), 'R2': -0.09003618974869276, 'dMAE_vs_zero_lo': np.float64(-0.26491273415731853), 'dMAE_vs_zero_hi': np.float64(-0.13011377649250788)}
